In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator


In [2]:
# 1. Setup paths and model type
sam_checkpoint = "sam_vit_h_4b8939.pth"
model_type = "vit_h"
device = "cuda" # Change to "cpu" if you don't have a GPU

# 2. Initialize SAM
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)
mask_generator = SamAutomaticMaskGenerator(sam)

/home/pranto/miniconda3/envs/ml/lib/python3.10/site-packages/segment_anything/build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


In [3]:

# 3. Load your image
image_path = 'download.jpeg' # Replace with your image file
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# 4. Generate Masks
print("Analyzing image... this may take a moment.")
masks = mask_generator.generate(image)

# 5. Process and Crop Each Segment
print(f"Detected {len(masks)} segments. Cropping...")

for i, mask_data in enumerate(masks):
    # 'segmentation' is a boolean mask of the object
    mask = mask_data['segmentation']
    
    # Create an image where everything outside the mask is black
    segmented_item = np.zeros_like(image)
    segmented_item[mask] = image[mask]
    
    # Get bounding box to crop the image to the specific item
    x, y, w, h = mask_data['bbox']
    crop = segmented_item[int(y):int(y+h), int(x):int(x+w)]
    
    # Convert back to BGR for saving with OpenCV
    crop_bgr = cv2.cvtColor(crop, cv2.COLOR_RGB2BGR)
    cv2.imwrite(f'food_item_{i}.png', crop_bgr)

print("Done! Check your folder for the cropped segments.")

Analyzing image... this may take a moment.


OutOfMemoryError: CUDA out of memory. Tried to allocate 1024.00 MiB. GPU 0 has a total capacity of 3.68 GiB of which 930.00 MiB is free. Including non-PyTorch memory, this process has 2.75 GiB memory in use. Of the allocated memory 2.59 GiB is allocated by PyTorch, and 71.29 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)